In [1]:
### CATS
### ETTH
### 3 epocas aproximadaemtne 10 segs
### 10 epocas aproximadamente 30 segs
### M4
## No termina smh
#Epoch: 1 cost time: 259.2893192768097
#iters: 17300, epoch: 10 | loss: 0.1217130

### YEARLY

### SMAPE 13.263
### MASE  2.967
### OWA   0.779

### FIX


import torch
import pandas as pd, numpy as np, glob
import os, shutil, subprocess

if os.path.exists('/content/CATS-main'):
    shutil.rmtree('/content/CATS-main')
!unzip -q /content/CATS-main.zip -d /content/
%cd /content/CATS-main

!pip install -q matplotlib pandas scikit-learn einops huggingface_hub

!grep -rl "np.Inf" --include="*.py" . | xargs sed -i 's/np\.Inf/np.inf/g' 2>/dev/null
!grep -rl "np.NaN" --include="*.py" . | xargs sed -i 's/np\.NaN/np.nan/g' 2>/dev/null

!unzip -o -q /content/m4_port.zip -d /content/m4_port
!cp /content/m4_port/m4.py             ./data/m4.py
!cp /content/m4_port/data_loader_m4.py ./data/data_loader_m4.py
!cp /content/m4_port/losses.py         ./utils/losses.py
!cp /content/m4_port/m4_summary.py     ./utils/m4_summary.py
!cp /content/m4_port/exp_short_term_forecasting.py ./exp/exp_short_term_forecasting.py

!sed -i 's/np.array(y))/np.array(y, dtype=np.float32))/; s/trues = np.array(y)/trues = np.array(y, dtype=np.float32)/' ./exp/exp_short_term_forecasting.py

import re
src = open('run.py').read()
if '--seasonal_patterns' not in src:
    src = src.replace(
        "    args = parser.parse_args()",
        "    parser.add_argument('--seasonal_patterns', type=str, default='Yearly', help='M4 subset')\n\n    args = parser.parse_args()", 1)
if "args.data == 'm4'" not in src:
    src = src.replace(
        "    Exp = Exp_Main\n",
        "    Exp = Exp_Main\n"
        "    if args.data == 'm4':\n"
        "        from exp.exp_short_term_forecasting import Exp_Short_Term_Forecast\n"
        "        Exp = Exp_Short_Term_Forecast\n", 1)
open('run.py','w').write(src)
print("run.py parcheado:", '--seasonal_patterns' in src, "| ruteo m4:", "args.data == 'm4'" in src)

!mkdir -p ./dataset/m4

subsets = {
    'Yearly':    dict(pl=6,  dm=64),
    'Quarterly': dict(pl=8,  dm=64),
    'Monthly':   dict(pl=18, dm=64),
    'Weekly':    dict(pl=13, dm=64),
    'Daily':     dict(pl=14, dm=64),
    'Hourly':    dict(pl=48, dm=128),
}

for sp, hp in subsets.items():
    print(f"\n{'='*60}\nCATS  ->  M4-{sp}  (pred_len={hp['pl']})\n{'='*60}")
    cmd = f"""python -u run.py \
      --is_training 1 \
      --root_path ./dataset/m4 \
      --data_path {sp} \
      --model_id m4_{sp} \
      --model CATS \
      --data m4 \
      --features S \
      --seasonal_patterns {sp} \
      --label_len 0 \
      --d_layers 3 \
      --dec_in 1 \
      --d_model {hp['dm']} \
      --d_ff 256 \
      --n_heads 8 \
      --QAM_end 0.2 \
      --batch_size 32 \
      --patch_len {hp['pl']} \
      --stride {hp['pl']} \
      --train_epochs 30 \
      --patience 10 \
      --loss SMAPE \
      --learning_rate 0.001 \
      --num_workers 2"""
    subprocess.run(cmd, shell=True)

print("\nCATS: 6 subconjuntos entrenados en /m4_results/CATS/")


/content/CATS-main
run.py parcheado: True | ruteo m4: True

CATS  ->  M4-Yearly  (pred_len=6)

CATS  ->  M4-Quarterly  (pred_len=8)

CATS  ->  M4-Monthly  (pred_len=18)

CATS  ->  M4-Weekly  (pred_len=13)

CATS  ->  M4-Daily  (pred_len=14)

CATS  ->  M4-Hourly  (pred_len=48)

CATS: 6 subconjuntos entrenados. Forecasts en ./m4_results/CATS/


In [2]:
!nvidia-smi

Thu Jun 18 03:47:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             35W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import os, glob
import numpy as np, pandas as pd
from data.m4 import M4Dataset, M4Meta

ROOT = './dataset/m4'
MODEL_DIR = './m4_results/CATS/'

NAIVE2_SMAPE = {'Yearly':16.342,'Quarterly':11.012,'Monthly':14.427,'Weekly':9.161,'Daily':3.045,'Hourly':18.383}
NAIVE2_MASE  = {'Yearly':3.974,'Quarterly':1.371,'Monthly':1.063,'Weekly':2.777,'Daily':3.278,'Hourly':2.395}

def smape(a, f):
    d = np.abs(a) + np.abs(f)
    return 200 * np.mean(np.where(d == 0, 0.0, np.abs(a - f) / d))

def mase(a, f, hist, sp):
    scale = np.mean(np.abs(hist[sp:] - hist[:-sp])) + 1e-8
    return np.mean(np.abs(a - f)) / scale

train_ds = M4Dataset.load(training=True,  dataset_file=ROOT)
test_ds  = M4Dataset.load(training=False, dataset_file=ROOT)

print(f"{'Subconjunto':12s} {'SMAPE':>8s} {'MASE':>8s} {'OWA':>8s} {'Series':>8s}")
for sp in ['Yearly','Quarterly','Monthly','Weekly','Daily','Hourly']:
    path = MODEL_DIR + sp + '_forecast.csv'
    if not os.path.exists(path):
        continue
    H = M4Meta.horizons_map[sp]
    FREQ = M4Meta.frequency_map[sp]
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    fc = pd.read_csv(path, index_col=0) if cols and cols[0] == 'id' else pd.read_csv(path)
    preds = fc.values.astype(np.float32)
    insample = [v[~np.isnan(v)] for v in train_ds.values[train_ds.groups == sp]]
    trues = np.array([v[~np.isnan(v)][:H] for v in test_ds.values[test_ds.groups == sp]], dtype=np.float32)
    n = min(len(preds), len(trues))
    smapes = [smape(trues[i], preds[i]) for i in range(n)]
    mases  = [mase(trues[i], preds[i], insample[i], FREQ) for i in range(n)]
    ms, mm = np.mean(smapes), np.mean(mases)
    owa = 0.5 * (ms / NAIVE2_SMAPE[sp] + mm / NAIVE2_MASE[sp])
    print(f"{sp:12s} {ms:8.3f} {mm:8.3f} {owa:8.3f} {n:8d}")

Subconjunto     SMAPE     MASE      OWA   Series
Yearly         13.468    3.050    0.796    23000
Quarterly      10.100    1.179    0.889    24000
Monthly        12.859    0.955    0.895    48000
Weekly         10.027    3.295    1.141      359
Daily           3.027    3.227    0.989     4227
Hourly         18.211    3.408    1.207      414


In [ ]:
### Fredformer
### ETTH
### 3 epocas 30 segs aprox
### 10 epocas 1 min aprox
### M4
###
###

import torch
import pandas as pd, numpy as np, glob
import os, shutil, subprocess

if os.path.exists('/content/Fredformer-main'):
    shutil.rmtree('/content/Fredformer-main')
!unzip -q /content/Fredformer-main.zip -d /content/
%cd /content/Fredformer-main

!pip install -q einops reformer-pytorch sktime torchinfo thop huggingface_hub

!grep -rl "np.Inf" --include="*.py" . | xargs sed -i 's/np\.Inf/np.inf/g' 2>/dev/null
!grep -rl "np.NaN" --include="*.py" . | xargs sed -i 's/np\.NaN/np.nan/g' 2>/dev/null

!unzip -o -q /content/fred_m4_port.zip -d /content/fred_m4_port
!mkdir -p ./data_provider
!cp /content/fred_m4_port/m4.py             ./data_provider/m4.py
!cp /content/fred_m4_port/data_loader_m4.py ./data_provider/data_loader_m4.py
!cp /content/fred_m4_port/losses.py         ./utils/losses.py
!cp /content/fred_m4_port/m4_summary.py     ./utils/m4_summary.py
!cp /content/fred_m4_port/exp_short_term_forecasting.py ./exp/exp_short_term_forecasting.py

!sed -i 's/from data\.m4/from data_provider.m4/g; s/from data\.data_loader_m4/from data_provider.data_loader_m4/g' ./exp/exp_short_term_forecasting.py
!sed -i 's/from data\.m4/from data_provider.m4/g' ./data_provider/data_loader_m4.py ./utils/m4_summary.py

!sed -i 's/np.array(y))/np.array(y, dtype=np.float32))/; s/trues = np.array(y)/trues = np.array(y, dtype=np.float32)/' ./exp/exp_short_term_forecasting.py
!sed -i 's/def train(self, setting):/def train(self, setting, args=None):/; s/def test(self, setting, test=0):/def test(self, setting, args=None, test=0):/' ./exp/exp_short_term_forecasting.py

import re
src = open('run_longExp.py').read()
if '--seasonal_patterns' not in src:
    src = src.replace("args = parser.parse_args()",
        "parser.add_argument('--seasonal_patterns', type=str, default='Yearly', help='M4 subset')\nargs = parser.parse_args()", 1)
if "args.data == 'm4'" not in src:
    src = src.replace("Exp = Exp_Main\n",
        "Exp = Exp_Main\n"
        "if args.data == 'm4':\n"
        "    from exp.exp_short_term_forecasting import Exp_Short_Term_Forecast\n"
        "    Exp = Exp_Short_Term_Forecast\n", 1)
open('run_longExp.py', 'w').write(src)
print("run_longExp.py parcheado:", '--seasonal_patterns' in src, "| ruteo m4:", "args.data == 'm4'" in src)

!mkdir -p ./dataset/m4

subsets = {
    'Yearly':6, 'Quarterly':8, 'Monthly':18, 'Weekly':13, 'Daily':14, 'Hourly':48
}

for sp, pl in subsets.items():
    dm = 128 if sp == 'Hourly' else 64
    print(f"\n{'='*60}\nFredformer  ->  M4-{sp}  (pred_len={pl})\n{'='*60}")
    cmd = f"""python -u run_longExp.py \
      --random_seed 2021 --is_training 1 --itr 1 \
      --root_path ./dataset/m4 --data_path {sp} \
      --model_id m4_{sp} --model Fredformer --data m4 --features S \
      --seasonal_patterns {sp} --label_len 0 --enc_in 1 \
      --e_layers 3 --d_model {dm} --d_ff 256 --n_heads 8 \
      --dropout 0.3 --fc_dropout 0.3 \
      --patch_len {pl} --stride {pl} \
      --cf_dim 128 --cf_depth 2 --cf_heads 8 --cf_mlp 96 --cf_head_dim 32 \
      --use_nys 0 --individual 0 \
      --batch_size 32 --train_epochs 30 --patience 10 \
      --loss SMAPE --learning_rate 0.001 --num_workers 2"""
    subprocess.run(cmd, shell=True)

print("\nFredformer: 6 subconjuntos entrenados en ./m4_results/Fredformer/")


shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/Fredformer-main
run_longExp.py parcheado: True | ruteo m4: True

Test M4 Fredformer (protocolo nativo)
Args in experiment:
Namespace(cf_dim=128, cf_drop=0.2, cf_depth=2, cf_heads=8, cf_mlp=96, cf_head_dim=32, cf_weight_decay=0, cf_p=1, use_nys=0, mlp_drop=0.3, ablation=0, random_seed=2021, is_training=1, model_id='m4_Yearly', model='Fredformer', data='m4', root_path='./dataset/m4', data_path='Yearly', features='S', target='OT', freq='h', checkpoints='./checkpoints/', seq_len=96, label_len=0, pred_len=96, fc_dropout=0.3, head_dropout=0.0, patch_len=6, stride=6, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, embed_type=0, enc_in=1, dec_in=7, c_out=7, d_model=64, mlp_hidden=64, n_heads=8, e_layers=3, d_layers=1, d_ff=256, moving_avg=25, factor=1, distil=True, dropout=0.3, embed='timeF', activation='gelu', outp

In [3]:
import os, glob
import pandas as pd
from huggingface_hub import hf_hub_download

ROOT = './dataset/m4'
MODEL_DIR = './m4_results/CATS/'

if not os.path.exists(os.path.join(ROOT, 'submission-Naive2.csv')):
    hf_hub_download(repo_id='thuml/Time-Series-Library',
                    filename='m4/submission-Naive2.csv',
                    repo_type='dataset', local_dir='./dataset',
                    local_dir_use_symlinks=False)
    print("Naive2 oficial descargado.")

for f in glob.glob(MODEL_DIR + '*_forecast.csv'):
    cols = pd.read_csv(f, nrows=0).columns.tolist()
    if cols and cols[0] == 'id':
        pd.read_csv(f, index_col=0).to_csv(f, index=False)
        print("normalizado:", os.path.basename(f))

from utils.m4_summary import M4Summary

m4 = M4Summary(MODEL_DIR, ROOT)
smape, owa, mape, mase = m4.evaluate()

print("\n===== RESULTADOS OFICIALES M4 =====")
print(f"{'Subconjunto':12s} {'SMAPE':>8s} {'MASE':>8s} {'OWA':>8s}")
for k in smape.keys():
    print(f"{k:12s} {smape[k]:8.3f} {mase[k]:8.3f} {owa[k]:8.3f}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


m4/submission-Naive2.csv:   0%|          | 0.00/23.4M [00:00<?, ?B/s]

Naive2 oficial descargado.
normalizado: Quarterly_forecast.csv
normalizado: Weekly_forecast.csv
normalizado: Yearly_forecast.csv
normalizado: Hourly_forecast.csv
normalizado: Daily_forecast.csv
normalizado: Monthly_forecast.csv


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (100000,) + inhomogeneous part.

In [4]:
import pandas as pd
a = pd.read_csv('/content/CATS-main/m4_results/CATS/Yearly_forecast.csv', index_col=0)
b = pd.read_csv('/content/Fredformer-main/m4_results/Fredformer/Yearly_forecast.csv', index_col=0)
print("¿Predicciones idénticas?", a.round(4).equals(b.round(4)))
print(a.head(2)); print(b.head(2))

FileNotFoundError: [Errno 2] No such file or directory: '/content/Fredformer-main/m4_results/Fredformer/Yearly_forecast.csv'

Para 10 epocas (GPU 4)

CATS 10 segs

test 2785
mse:0.371696412563324, mae:0.39527738094329834, rmse:0.6096690893173218

Fredformer 59 segs

test 2785
mse:0.3760049641132355, mae:0.39402705430984497, rse:0.5814498066902161





CATS: Mejor MSE, Entrenamiento veloz

Fredformer: Mejor MAE,